# Evaluation of the effect of RAG and Chain-of-Thought on the performance of large language models

In [1]:
#This notebook will evaluate different answer generation functions on the mmlu dataset, summarizing the results and outputting them to CSV files.

import pandas as pd
import import_datasets
import general_utils
from generate_answer_cot import generate_answer_cot
#Import other answer generation functions as needed
from evaluate_model import evaluate_model
from rag import rag_pipeline
from generate_answer_rag import generate_answer_rag
from generate_answer_std import generate_answer_std

client = general_utils.load_gemini_client()



/Users/skysheng/miniforge3/envs/rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/skysheng/miniforge3/envs/rag/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


## RAG Implementation

> Warning: takes 30 minutes to run!

In [2]:
# build the rag pipeline to vectorize concept net
file_path = "../input_data/concept_net.txt"
vector_store = rag_pipeline(file_path, split_by="sentence", chunk_size=1000, batch_processing_size=700)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/skysheng/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Total splits: 1
Last batch!


In [ ]:
#Generate 100 samples from the mmlu datasets
dataset_names = ['anatomy', 'formal_logic', 'moral_scenarios']
datasets = import_datasets.import_datasets(dataset_names)
df_anatomy = datasets['anatomy']
df_formal_logic = datasets['formal_logic']
df_moral_scenarios = datasets['moral_scenarios']

Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 3360.28 examples/s]


In [55]:
#Evaluate the accuracy of each model on each dataset. Store the results in dataframes.
#Evaluate the standard model performance on each dataset
accuracy_anatomy_std, results_df_anatomy_std = evaluate_model(df_anatomy, generate_answer_std, client=client)
accuracy_formal_logic_std, results_df_formal_logic_std = evaluate_model(df_formal_logic, generate_answer_std, client=client)
accuracy_moral_scenarios_std, results_df_moral_scenarios_std = evaluate_model(df_moral_scenarios, generate_answer_std, client=client)


Error
Error
Error


In [62]:

import os
def save_df_to_csv(df, file_name, folder_path="../results"):
     """
     Save a dataframe to a csv file.
     Args:
         df (pd.DataFrame): The dataframe to save.
         file_name (str): The name of the file to save.
         folder_path (str): The path to the folder to save the file.
     """
     df.to_csv(os.path.join(folder_path, file_name), index=False)
save_df_to_csv(results_df_anatomy_std, "results_df_anatomy_std.csv")
save_df_to_csv(results_df_formal_logic_std, "results_df_formal_logic_std.csv")
save_df_to_csv(results_df_moral_scenarios_std, "results_df_moral_scenarios_std.csv")

In [61]:
general_utils.save_df_to_csv(results_df_anatomy_std, "results_df_anatomy_std.csv")
general_utils.save_df_to_csv(results_df_formal_logic_std, "results_df_formal_logic_std.csv")
general_utils.save_df_to_csv(results_df_moral_scenarios_std, "results_df_moral_scenarios_std.csv")

AttributeError: module 'general_utils' has no attribute 'save_df_to_csv'

In [ ]:
#Evaluate the cot model performance on each dataset
accuracy_anatomy_cot, results_df_anatomy_cot = evaluate_model(df_anatomy, generate_answer_cot, client=client)
accuracy_formal_logic_cot, results_df_formal_logic_cot = evaluate_model(df_formal_logic, generate_answer_cot, client=client)
accuracy_moral_scenarios_cot, results_df_moral_scenarios_cot = evaluate_model(df_moral_scenarios, generate_answer_cot, client=client)

In [ ]:
# WARNING: take ~ 15 minutes to run: Evaluate the RAG model performance on each dataset
accuracy_anatomy_rag, results_df_anatomy_rag = evaluate_model(df_anatomy, generate_answer_rag, client=client, vector_store=vector_store, k=1)
accuracy_formal_logic_rag, results_df_formal_logic_rag = evaluate_model(df_formal_logic, generate_answer_rag, client=client, vector_store=vector_store, k=1)
accuracy_moral_scenarios_rag, results_df_moral_scenarios_rag = evaluate_model(df_moral_scenarios, generate_answer_rag, client=client, vector_store=vector_store, k=1)

In [ ]:
#Combine results for each model into dataframes:
results_df_std = pd.concat([results_df_anatomy_std, results_df_formal_logic_std, results_df_moral_scenarios_std], ignore_index=True)
results_df_cot = pd.concat([results_df_anatomy_cot, results_df_formal_logic_cot, results_df_moral_scenarios_cot], ignore_index=True)
results_df_rag = pd.concat([results_df_anatomy_rag, results_df_formal_logic_rag, results_df_moral_scenarios_rag], ignore_index=True)

In [ ]:
#create a dataframe to summarize the results
results_summary = pd.DataFrame({
    'Dataset': ['Standard', 'CoT', 'RAG'],
    'Anatomy': [accuracy_anatomy_std, accuracy_anatomy_cot, accuracy_anatomy_rag],
    'Formal Logic': [accuracy_formal_logic_std, accuracy_formal_logic_cot, accuracy_formal_logic_rag],
    'Moral Scenarios': [accuracy_moral_scenarios_std, accuracy_moral_scenarios_cot, accuracy_moral_scenarios_rag]
}) 
print(results_summary)

results_summary.to_csv('model_performance_summary.csv', index=False)

    Dataset  Anatomy  Formal Logic  Moral Scenarios
0  Standard     0.35          0.13             0.20
1       CoT     0.00          0.00             0.00
2       RAG     0.17          0.26             0.22


In [ ]:
#Generate a dataframe with 20 randomly selected errors for each model:
def get_error_samples(results_df, model_name):
    errors_df = results_df[results_df['correct'] == False]
    sample_errors_df = errors_df.sample(n=min(20, len(errors_df)), random_state=42)
    sample_errors_df['model'] = model_name
    return sample_errors_df

#Generate a csv file containing at most 20 randomly selected errors for each model
error_samples_std = get_error_samples(results_df_std, 'Standard')
error_samples_cot = get_error_samples(results_df_cot, 'CoT')
error_samples_rag = get_error_samples(results_df_rag, 'RAG')
error_samples_df = pd.concat([error_samples_std, error_samples_cot, error_samples_rag], ignore_index=True)
error_samples_df.to_csv('model_error_samples.csv', index=False)